# Toy Regression - Google Colab doğrulaması

Yalnızca bu notebook'u Colab'a yükleyin. **Runtime > Change runtime type > GPU**
(varsa T4) seçip **Runtime > Run all** uygulayın. Drive bağlama, GitHub clone,
başka dosya yükleme veya Windows `.venv` taşıma gerekmez.

Notebook sekiz Python kaynağını ve yerel deneyin sabit 20 noktalı CSV'sini içerir.
02-06 temel deneyleri, 07 üç seed deneyini ve 08 tanı analizini sırayla çalıştırır.
01 veri üretim kaynağı da gömülüdür; aynı deney girdilerini korumak için CSV yeniden
üretilmez. Mevcut mimari, 3000 epoch, learning rate, seed ve M değerleri korunur.

**Bu dosya henüz gerçek Colab üzerinde denenmedi.** Yerel Run All kontrolü tüm
hücreleri eğitim kapalı modda yürütür. Gerçek Colab Run All, modelleri yeni ve
izole bir çalışma alanında eğitir; yerel sonuçları kullanmaz veya üzerine yazmaz.


In [ ]:
import os
import sys
import importlib.util
from pathlib import Path

# Must precede torch/CUDA initialization; unchanged deterministic setup for step 07.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ModuleNotFoundError, ImportError, ValueError):
    IN_COLAB = False
VALIDATION_ONLY = os.environ.get('TOY_COLAB_VALIDATE_ONLY') == '1' or not IN_COLAB
ALLOW_TRAINING = IN_COLAB and not VALIDATION_ONLY
print('Mode:', 'COLAB TRAINING' if ALLOW_TRAINING else 'LOCAL COMPATIBILITY ONLY - NO TRAINING')


## Ortam ve bağımlılıklar
Colab'ın hazır CUDA destekli PyTorch paketi korunur. Yalnız eksik NumPy veya
Matplotlib kurulabilir; torch/torchvision, Windows CUDA wheel'i veya eski `.venv`
kurulmaz. GPU yoksa eğitim başlamadan açık hata verilir.


In [ ]:
import subprocess
import inspect

missing = []
for module, requirement in [('numpy', 'numpy>=1.26,<3'), ('matplotlib', 'matplotlib>=3.8,<4')]:
    if importlib.util.find_spec(module) is None:
        missing.append(requirement)
if missing:
    if VALIDATION_ONLY:
        raise RuntimeError(f'Local validation dependencies are missing: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
if 'weights_only' not in inspect.signature(torch.load).parameters:
    raise RuntimeError('Use a clean current Colab runtime with a supported preinstalled PyTorch')
if ALLOW_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('GPU is unavailable. Select Runtime > Change runtime type > GPU, then Run all')
if VALIDATION_ONLY:
    # Defense in depth: an accidental training call must fail, even on a local GPU.
    def _deny_local_optimizer_step(*args, **kwargs):
        raise RuntimeError('Optimizer steps are prohibited in local compatibility validation')
    torch.optim.Adam.step = _deny_local_optimizer_step
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('NumPy:', np.__version__, '| Matplotlib:', matplotlib.__version__)
print('GPU detected:', torch.cuda.is_available(), '| training enabled:', ALLOW_TRAINING)


## Gömülü kaynakları yeni çalışma alanına aç
Her Run All yeni `/content/toy_regression_colab_runs/run_*` dizini oluşturur.
Hiçbir eski `outputs/` klasörü, checkpoint veya Drive sonucu kopyalanmaz.
Yerel uyumluluk modu işletim sisteminin geçici dizinini kullanır.


In [ ]:
import base64
import gzip
import hashlib
import json
import tempfile

PAYLOAD = '...'
EXPECTED_BUNDLE_SHA256 = '5dcbe6f9dae216c64366129e2c5ef335dadf20f73572dbf9391d1b67b9172f78'
bundle_bytes = base64.b64decode(PAYLOAD, validate=True)
if hashlib.sha256(bundle_bytes).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise AssertionError('Notebook bundle checksum mismatch')
bundle = json.loads(gzip.decompress(bundle_bytes))
if set(bundle['assets']) != set(bundle['sha256']):
    raise AssertionError('Bundle manifest mismatch')
base = Path('/content/toy_regression_colab_runs') if IN_COLAB else Path(tempfile.gettempdir()) / 'toy_regression_colab_runs'
base.mkdir(parents=True, exist_ok=True)
RUN_ROOT = Path(tempfile.mkdtemp(prefix='run_', dir=base)).resolve()
WORK = RUN_ROOT / 'toy_regression'
WORK.mkdir()
for relative, encoded in bundle['assets'].items():
    if relative.startswith('/') or '\\' in relative or ':' in relative or '..' in Path(relative).parts:
        raise ValueError('Unsafe embedded path')
    target = WORK / relative
    if not target.resolve().is_relative_to(WORK):
        raise ValueError('Bundle path escaped sandbox')
    content = base64.b64decode(encoded, validate=True)
    if hashlib.sha256(content).hexdigest() != bundle['sha256'][relative]:
        raise AssertionError(f'Embedded file changed: {relative}')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)

spec = importlib.util.spec_from_file_location('_toy_colab_runtime', WORK / 'colab_runtime.py')
runtime_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime_module)
run = runtime_module.ColabRun(RUN_ROOT, bundle['sha256'], allow_training=ALLOW_TRAINING)
print('Isolated run:', RUN_ROOT)


In [ ]:
preflight = run.preflight()
assert preflight['training_samples'] == 20
assert preflight['scientific_config']['run_seeds'] == [42, 123, 2026]
assert preflight['scientific_config']['epochs'] == 3000


In [ ]:
import csv
from IPython.display import display, Image
with (WORK / 'outputs/toy_training_data.csv').open(newline='', encoding='utf-8-sig') as stream:
    samples = list(csv.DictReader(stream))
x = np.array([float(row['x']) for row in samples])
y = np.array([float(row['y']) for row in samples])
grid = np.linspace(-6, 6, 400)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, grid**3, label='y = x^3')
ax.scatter(x, y, c='crimson', label='Original 20 fixed training points')
ax.set(xlabel='x', ylabel='y', title='Fixed local training data copied byte-for-byte')
ax.legend()
fig.tight_layout()
fig.savefig(WORK / 'outputs/toy_dataset_reference.png', dpi=150)
plt.close(fig)
display(Image(filename=str(WORK / 'outputs/toy_dataset_reference.png')))


## 02 - Tek MSE modeli
Kaynak dosya: `02_train_mse.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('02_train_mse.py')


## 03 - Beş üyeli MSE ensemble
Kaynak dosya: `03_train_mse_ensemble.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('03_train_mse_ensemble.py')


## 04 - Tek Gaussian NLL modeli
Kaynak dosya: `04_train_gaussian_nll.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('04_train_gaussian_nll.py')


## 05 - Gaussian NLL + adversarial training
Kaynak dosya: `05_train_gaussian_nll_at.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('05_train_gaussian_nll_at.py')


## 06 - Beş üyeli Gaussian deep ensemble
Kaynak dosya: `06_train_deep_ensemble.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('06_train_deep_ensemble.py')


## 07 - Üç bağımsız seed: 42, 123, 2026
Kaynak dosya: `07_multi_seed_experiments.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('07_multi_seed_experiments.py')


## 08 - Eğitimsiz belirsizlik tanısı
Kaynak dosya: `08_diagnose_uncertainty.py`. CLI epoch/seed override'ı kullanılmaz.


In [ ]:
run.run_step('08_diagnose_uncertainty.py')


## Çıktı doğrulama ve arşiv
Gerçek Colab koşusunda 49 model dosyası (13 temel model + üç seed'de 36 model),
48 seed/bölge metrik satırı ve 16 bölge özeti beklenir. Hata olursa Run All durur;
eksik eğitim sonuçları tamamlanmış gibi gösterilmez.


In [ ]:
ARCHIVE = run.finish()
print('Real Colab training completed:', run.report['real_colab_training_completed'])
print('Manifest:', RUN_ROOT / 'run_manifest.json')


In [ ]:
if ALLOW_TRAINING:
    for relative in [
        'toy_figure1_all_4_panels.png',
        '07_seed_experiments/aggregate/figures/summary_metrics.png',
        '08_uncertainty_diagnostics/figures/left_ood_intervals.png',
    ]:
        display(Image(filename=str(WORK / 'outputs' / relative)))
    with (WORK / 'outputs/07_seed_experiments/aggregate/metrics/summary_mean_sd.csv').open(newline='', encoding='utf-8') as stream:
        for row in csv.DictReader(stream):
            print(row)
else:
    print('Training figures and scores are intentionally unavailable in local validation')


In [ ]:
if ALLOW_TRAINING:
    from google.colab import files
    files.download(str(ARCHIVE))
else:
    print('No Colab download: local archive contains setup/validation only, not trained results')
print('Colab files are temporary. Keep the downloaded ZIP and save this executed notebook.')
